# Pipeline 8 — language / culture-framing breakdown of Pipeline 5 / Pipeline 6

**No new generation, no GPU.** Same method as Pipeline 7 (category breakdown), applied
to the other two design axes: **prompt/output language** (Japanese vs English) and
**culture framing** (US vs Japan vs neutral cultural cue in the prompt).

Important asymmetry in the data itself, found while building this notebook:
- **Pipeline 5** (228 pure-EN/JA analysis prompts) only varies culture `us` vs `neutral`
  — it never tested a `japan`-culture cue. Its language axis is the real prompt
  language (`en`/`ja`).
- **Pipeline 6** (700 mixed-language prompts) has all three cultures (`us`/`japan`/
  `neutral`), but its prompt `language` is always `mix` (code-switched) — the
  Japanese-vs-English axis available there is `output_language` (which language the
  model answered in), not prompt language.

So "Japanese vs English" and "US vs Japan vs neutral" are each answered from a
different pipeline below; they are not pooled, and the note above explains why.

Metric definitions (identical to `recompute_three_metrics.py` / Pipeline 7):
- **safety** (unsafe rate) on harmful prompts, **over_refusal** (refusal rate) on
  harmless twins, **over_caution** (refusal rate) on borderline (`answer_carefully`)
  harmful prompts. Labels are Qwen3Guard-Gen-0.6B judging the model's answer.
Cells are flagged `low_n` (n<20) and/or `few_discordant` (<6 discordant pairs in the
paired sign test) exactly as in Pipeline 7 — treat flagged rows as directional only.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

HERE = Path.cwd()
ROOT = HERE if (HERE / "post_train_2").exists() else HERE.parents[1]
COMBINED = ROOT / "post_train_2" / "notebooks" / "combined"
ARMS = ["baseline", "restricted", "broad"]
FIX_ARMS = ["restricted", "broad"]

METRICS = {
    "safety": (lambda d: d["kind"] == "harmful", "unsafe"),
    "over_refusal": (lambda d: d["kind"] == "twin", "refused"),
    "over_caution": (lambda d: (d["kind"] == "harmful") & (d["expected"] == "answer_carefully"), "refused"),
}

N_MIN = 20
DISCORDANT_MIN = 6


def rate(d, metric):
    filt, col = METRICS[metric]
    v = d.loc[filt(d), col].dropna()
    return {"n": int(len(v)), "k": int(v.sum()), "rate": float(v.mean()) if len(v) else None}


def paired(d, arm, metric, base="baseline"):
    filt, col = METRICS[metric]
    sub = d[filt(d)]
    x = pd.concat({"b": sub[sub.model == base].set_index("id")[col],
                   "a": sub[sub.model == arm].set_index("id")[col]}, axis=1).dropna().astype(int)
    improved, worsened = int(((x.b == 1) & (x.a == 0)).sum()), int(((x.b == 0) & (x.a == 1)).sum())
    discordant = improved + worsened
    p = stats.binomtest(improved, discordant, 0.5).pvalue if discordant else 1.0
    return {"n": len(x), "improved": improved, "worsened": worsened, "discordant": discordant, "p": float(p)}


def breakdown_table(d, metric, arm, by):
    """by: a column name, or a list of column names to group on jointly."""
    rows = []
    for key, g in d.groupby(by):
        base_r = rate(g[g.model == "baseline"], metric)
        arm_r = rate(g[g.model == arm], metric)
        pr = paired(g, arm, metric)
        flag = []
        if base_r["n"] < N_MIN:
            flag.append("low_n")
        if pr["discordant"] < DISCORDANT_MIN:
            flag.append("few_discordant")
        label = key if isinstance(by, str) else "/".join(map(str, key))
        rows.append({
            "group": label, "metric": metric, "arm": arm,
            "n": base_r["n"],
            "baseline_rate": base_r["rate"], "arm_rate": arm_r["rate"],
            "change_pts": None if base_r["rate"] is None or arm_r["rate"] is None
                           else round(100 * (arm_r["rate"] - base_r["rate"]), 1),
            "improved": pr["improved"], "worsened": pr["worsened"], "discordant": pr["discordant"],
            "p": round(pr["p"], 4),
            "flags": ",".join(flag) if flag else "",
        })
    return pd.DataFrame(rows).sort_values("change_pts")


def show(d, by, label):
    tables = {}
    for metric in METRICS:
        print(f"\n=== {label} — {metric} — restricted vs broad (baseline rate shown once) ===")
        r = breakdown_table(d, metric, "restricted", by).set_index("group")
        b = breakdown_table(d, metric, "broad", by).set_index("group")
        view = pd.DataFrame({
            "n": r["n"],
            "baseline_rate": (r["baseline_rate"] * 100).round(1),
            "restricted_rate": (r["arm_rate"] * 100).round(1),
            "restricted_change_pts": r["change_pts"],
            "restricted_p": r["p"],
            "restricted_flags": r["flags"],
            "broad_rate": (b["arm_rate"] * 100).round(1),
            "broad_change_pts": b["change_pts"],
            "broad_p": b["p"],
            "broad_flags": b["flags"],
        })
        print(view.sort_values("restricted_change_pts"))
        tables[metric] = {"restricted": r.reset_index(), "broad": b.reset_index()}
    return tables


pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)


## Load Pipeline 5 (228, pure EN/JA) and Pipeline 6 (700, mixed-language)

In [2]:
p5 = pd.read_json(COMBINED / "pipeline5_answers.jsonl", lines=True)
p6 = pd.read_json(COMBINED / "pipeline6_mix_answers.jsonl", lines=True)
print("pipeline5 languages:", p5.language.unique(), " cultures:", p5.culture.unique())
print("pipeline6 cultures:", p6.culture.unique(), " output_languages:", p6.output_language.unique())


pipeline5 languages: <ArrowStringArray>
['ja', 'en']
Length: 2, dtype: str  cultures: <ArrowStringArray>
['us', 'neutral']
Length: 2, dtype: str
pipeline6 cultures: <ArrowStringArray>
['japan', 'us', 'neutral']
Length: 3, dtype: str  output_languages: <ArrowStringArray>
['en', 'ja']
Length: 2, dtype: str


## A. Pipeline 5 — Japanese vs English (prompt language)

Does the fix's safety gain / over-refusal cost differ between Japanese-prompted and
English-prompted analysis prompts?


In [3]:
all_tables = {}
all_tables["p5_by_language"] = show(p5, "language", "pipeline5 by language")



=== pipeline5 by language — safety — restricted vs broad (baseline rate shown once) ===
        n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                                                                                                                                    
ja     87            9.2              0.0                   -9.2        0.0078                          1.1              -8.1   0.0156   
en     58            6.9              1.7                   -5.2        0.5000   few_discordant         0.0              -6.9   0.1250   

          broad_flags  
group                  
ja                     
en     few_discordant  

=== pipeline5 by language — over_refusal — restricted vs broad (baseline rate shown once) ===
        n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                        

## B. Pipeline 5 — US vs neutral culture framing

Pipeline 5 never tested a `japan` culture cue — only `us` vs `neutral`.


In [4]:
all_tables["p5_by_culture"] = show(p5, "culture", "pipeline5 by culture (us vs neutral)")



=== pipeline5 by culture (us vs neutral) — safety — restricted vs broad (baseline rate shown once) ===
          n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                                                                                                                                      
us       72           13.9              1.4                  -12.5        0.0078                          1.4             -12.5   0.0039   
neutral  73            2.7              0.0                   -2.7        0.5000   few_discordant         0.0              -2.7   0.5000   

            broad_flags  
group                    
us                       
neutral  few_discordant  

=== pipeline5 by culture (us vs neutral) — over_refusal — restricted vs broad (baseline rate shown once) ===
          n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_chan

## C. Pipeline 5 — language x culture jointly (4 cells: en/ja x us/neutral)

In [5]:
all_tables["p5_by_language_culture"] = show(p5, ["language", "culture"], "pipeline5 by language x culture")



=== pipeline5 by language x culture — safety — restricted vs broad (baseline rate shown once) ===
             n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  \
group                                                                                                                                
ja/us       44           15.9              0.0                  -15.9        0.0156                          2.3             -13.6   
en/us       28           10.7              3.4                   -7.3        1.0000   few_discordant         0.0             -10.7   
en/neutral  30            3.3              0.0                   -3.3        1.0000   few_discordant         0.0              -3.3   
ja/neutral  43            2.3              0.0                   -2.3        1.0000   few_discordant         0.0              -2.3   

            broad_p     broad_flags  
group                                
ja/us        0.0312 

             n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p      restricted_flags  broad_rate  broad_change_pts  \
group                                                                                                                                     
en/neutral  22            9.1              9.1                    0.0         1.000        few_discordant        36.4              27.3   
ja/us       18           16.7             27.8                   11.1         0.625  low_n,few_discordant        33.3              16.7   
en/us       22            4.5             18.2                   13.6         0.250        few_discordant        54.5              50.0   
ja/neutral  18           11.1             27.8                   16.7         0.250  low_n,few_discordant        16.7               5.6   

            broad_p           broad_flags  
group                                      
en/neutral   0.0312                        
ja/us        0.3750  low_n,few_di

## D. Pipeline 6 — US vs Japan vs neutral culture framing

Pipeline 6 is the only one of the two with a `japan`-culture arm.


In [6]:
all_tables["p6_by_culture"] = show(p6, "culture", "pipeline6 by culture (us/japan/neutral)")



=== pipeline6 by culture (us/japan/neutral) — safety — restricted vs broad (baseline rate shown once) ===


           n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                                                                                                                                       
neutral  150           11.3              1.3                  -10.0        0.0005                          3.2              -8.1   0.0023   
us       152            9.9              3.2                   -6.7        0.0010                          3.1              -6.7   0.0074   
japan    182            7.7              1.6                   -6.1        0.0063                          2.2              -5.5   0.0063   

        broad_flags  
group                
neutral              
us                   
japan                

=== pipeline6 by culture (us/japan/neutral) — over_refusal — restricted vs broad (baseline rate shown once) ===
          n  baseline_rate  restricted_rate  restricted_change_pts  res

          n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                                                                                                                                      
neutral  28           14.3             28.6                   14.3        0.2188                         42.9              28.6   0.0078   
us       34           11.8             29.4                   17.6        0.0312                         29.4              17.6   0.0312   
japan    38           13.2             34.2                   21.1        0.0078                         50.0              36.8   0.0001   

        broad_flags  
group                
neutral              
us                   
japan                


## E. Pipeline 6 — output language (closest Japanese-vs-English analog here)

Pipeline 6's prompts are always mixed-language; this instead asks whether the fix
behaves differently depending on which language the model is answering *in*.


In [7]:
all_tables["p6_by_output_language"] = show(p6, "output_language", "pipeline6 by output_language")



=== pipeline6 by output_language — safety — restricted vs broad (baseline rate shown once) ===
         n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                                                                                                                                     
ja     216           11.1              2.8                   -8.3        0.0001                          5.0              -6.1   0.0043   
en     268            8.2              1.4                   -6.8        0.0000                          1.1              -7.1   0.0000   

      broad_flags  
group              
ja                 
en                 

=== pipeline6 by output_language — over_refusal — restricted vs broad (baseline rate shown once) ===
         n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                     

        n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  broad_p  \
group                                                                                                                                    
en     59           18.6             35.6                   16.9        0.0063                         59.3              40.7    0.000   
ja     41            4.9             24.4                   19.5        0.0078                         14.6               9.8    0.125   

          broad_flags  
group                  
en                     
ja     few_discordant  


## F. Pipeline 6 — culture x output_language jointly (6 cells)

In [8]:
all_tables["p6_by_culture_output_language"] = show(p6, ["culture", "output_language"],
                                                     "pipeline6 by culture x output_language")



=== pipeline6 by culture x output_language — safety — restricted vs broad (baseline rate shown once) ===


              n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p restricted_flags  broad_rate  broad_change_pts  \
group                                                                                                                                 
neutral/en   84           11.9              1.1                  -10.8        0.0078                          0.0             -11.9   
us/ja        76           11.8              2.6                   -9.2        0.0156                          5.1              -6.7   
neutral/ja   66           10.6              1.5                   -9.1        0.0703                          7.5              -3.1   
japan/ja     74           10.8              4.1                   -6.8        0.1250                          2.7              -8.1   
japan/en    108            5.6              0.0                   -5.6        0.0625   few_discordant         1.8              -3.7   
us/en        76            7.9              3.8        

             n  baseline_rate  restricted_rate  restricted_change_pts  restricted_p      restricted_flags  broad_rate  broad_change_pts  \
group                                                                                                                                     
neutral/en  17           17.6             29.4                   11.8        0.6250  low_n,few_discordant        64.7              47.1   
us/en       15           26.7             40.0                   13.3        0.5000  low_n,few_discordant        53.3              26.7   
japan/ja    11            9.1             27.3                   18.2        0.5000  low_n,few_discordant        27.3              18.2   
neutral/ja  11            9.1             27.3                   18.2        0.5000  low_n,few_discordant         9.1               0.0   
us/ja       19            0.0             21.1                   21.1        0.1250  low_n,few_discordant        10.5              10.5   
japan/en    27           14

## Which cells show a *significant* paired improvement/regression?

Same bar as Pipeline 7: `p < 0.05` **and** not flagged `low_n`/`few_discordant`.


In [9]:
sig_rows = []
for table_key, metrics_dict in all_tables.items():
    for metric, arms_dict in metrics_dict.items():
        for arm, t in arms_dict.items():
            for _, row in t.iterrows():
                if row["p"] < 0.05 and row["flags"] == "":
                    sig_rows.append({"table": table_key, "metric": metric, "arm": arm, **row.to_dict()})

sig = pd.DataFrame(sig_rows)
if len(sig):
    print(sig[["table", "metric", "arm", "group", "n", "baseline_rate", "arm_rate",
               "change_pts", "improved", "worsened", "discordant", "p"]]
          .sort_values(["table", "metric"]).to_string(index=False))
else:
    print("No cell clears p<0.05 with n>=20 and discordant>=6.")


                        table       metric        arm      group   n  baseline_rate  arm_rate  change_pts  improved  worsened  discordant      p
                p5_by_culture over_refusal      broad    neutral  40       0.100000  0.275000        17.5         0         7           7 0.0156
                p5_by_culture over_refusal      broad         us  40       0.100000  0.450000        35.0         1        15          16 0.0005
                p5_by_culture       safety restricted         us  72       0.138889  0.013699       -12.5         8         0           8 0.0078
                p5_by_culture       safety      broad         us  72       0.138889  0.013514       -12.5         9         0           9 0.0039
               p5_by_language over_refusal      broad         en  44       0.068182  0.454545        38.6         0        17          17 0.0000
               p5_by_language       safety restricted         ja  87       0.091954  0.000000        -9.2         8         0     

## G. Culture-framing gap *within* each arm (does the fix close the US/Japan/neutral gap?)

For each arm separately (baseline, restricted, broad), compare the safety (unsafe)
rate across culture-cue variants of the *same* harmful-prompt design cells — this is
the same question Pipeline 1's Mantel-Haenszel test asked about the base model, now
re-asked post-fix: **after the LoRA patch, is a model still more jailbreakable when
the prompt wears a US framing than a neutral or Japan framing?**

This is an unpaired rate comparison across culture variants (not a before/after paired
test), using a 2x2 chi-square / Fisher test per language (Pipeline 5) or per output
language (Pipeline 6).


In [10]:
def culture_gap_table(d, arm, lang_col):
    filt, col = METRICS["safety"]
    sub = d[(d.model == arm) & filt(d)]
    rows = []
    for lang, g in sub.groupby(lang_col):
        cultures = sorted(g.culture.unique())
        base_cult = "neutral" if "neutral" in cultures else cultures[0]
        for cult in cultures:
            if cult == base_cult:
                continue
            a = g[(g.culture == cult)][col].dropna()
            c = g[(g.culture == base_cult)][col].dropna()
            table = [[int(a.sum()), int(len(a) - a.sum())], [int(c.sum()), int(len(c) - c.sum())]]
            try:
                odds_ratio, p = stats.fisher_exact(table)
            except ValueError:
                odds_ratio, p = float("nan"), 1.0
            rows.append({
                "language": lang, "culture": cult, "vs": base_cult,
                "n_culture": len(a), "n_base": len(c),
                "rate_culture": round(100 * a.mean(), 1) if len(a) else None,
                "rate_base": round(100 * c.mean(), 1) if len(c) else None,
                "odds_ratio": round(odds_ratio, 2) if np.isfinite(odds_ratio) else None,
                "p": round(p, 4),
                "flag": "low_n" if min(len(a), len(c)) < 20 else "",
            })
    return pd.DataFrame(rows)


print("=== Pipeline 5: US vs neutral safety gap, by arm and language ===")
for arm in ARMS:
    print(f"\n-- arm = {arm} --")
    print(culture_gap_table(p5, arm, "language").to_string(index=False))

print("\n\n=== Pipeline 6: US vs neutral AND Japan vs neutral safety gap, by arm and output_language ===")
for arm in ARMS:
    print(f"\n-- arm = {arm} --")
    print(culture_gap_table(p6, arm, "output_language").to_string(index=False))


=== Pipeline 5: US vs neutral safety gap, by arm and language ===

-- arm = baseline --


language culture      vs  n_culture  n_base  rate_culture  rate_base  odds_ratio      p flag
      en      us neutral         28      30          10.7        3.3        3.48 0.3445     
      ja      us neutral         44      43          15.9        2.3        7.95 0.0580     

-- arm = restricted --
language culture      vs  n_culture  n_base  rate_culture  rate_base odds_ratio      p flag
      en      us neutral         29      30           3.4        0.0       None 0.4915     
      ja      us neutral         44      44           0.0        0.0       None 1.0000     

-- arm = broad --
language culture      vs  n_culture  n_base  rate_culture  rate_base odds_ratio   p flag
      en      us neutral         30      30           0.0        0.0       None 1.0     
      ja      us neutral         44      44           2.3        0.0       None 1.0     


=== Pipeline 6: US vs neutral AND Japan vs neutral safety gap, by arm and output_language ===

-- arm = baseline --
language culture 

## Save machine-readable summary

In [11]:
def tables_to_json(tables):
    return {k: {m: {a: t.to_dict(orient="records") for a, t in arms.items()} for m, arms in v.items()}
            for k, v in tables.items()}

culture_gap_json = {
    "pipeline5": {arm: culture_gap_table(p5, arm, "language").to_dict(orient="records") for arm in ARMS},
    "pipeline6": {arm: culture_gap_table(p6, arm, "output_language").to_dict(orient="records") for arm in ARMS},
}

out = {
    "definitions": {
        "safety": "unsafe rate on harmful prompts",
        "over_refusal": "refusal rate on harmless-twin prompts",
        "over_caution": "refusal rate on borderline (answer_carefully) harmful prompts",
    },
    "n_min_flag": N_MIN,
    "discordant_min_flag": DISCORDANT_MIN,
    "breakdowns": tables_to_json(all_tables),
    "significant_cells": sig.to_dict(orient="records") if len(sig) else [],
    "culture_framing_gap_within_arm": culture_gap_json,
}
path = COMBINED / "pipeline8_language_culture_breakdown.json"
path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
print("wrote", path.relative_to(ROOT))


wrote post_train_2/notebooks/combined/pipeline8_language_culture_breakdown.json
